# Train hadith-scholar Model (Google Colab)

Fine-tune Phi-4-mini on hadith/Quran data using Unsloth + QLoRA on a free T4 GPU.

**Prerequisites:**
- Upload `data/train.jsonl` (generated by `scripts/prepare_training_data.py`)

**Runtime:** Runtime → Change runtime type → **T4 GPU**

In [ ]:
# Cell 1: Install Unsloth (Colab-optimized)
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
from torch import __version__; from packaging.version import Version as V
xformers = "xformers==0.0.27" if V(__version__) < V("2.4.0") else "xformers"
!pip install --no-deps {xformers} trl peft accelerate bitsandbytes triton

In [ ]:
# Cell 2: Upload training data
from google.colab import files
import os, json

if not os.path.exists("train.jsonl"):
    print("Upload train.jsonl:")
    uploaded = files.upload()
else:
    print("train.jsonl already exists")

with open("train.jsonl") as f:
    lines = f.readlines()
sample = json.loads(lines[0])
assert "messages" in sample, "ERROR: train.jsonl must have 'messages' field"
print(f"Training examples: {len(lines)}")
print(f"Roles: {[m['role'] for m in sample['messages']]}")

In [ ]:
# Cell 3: Load model
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-4-mini-instruct",
    max_seq_length=2048,
    load_in_4bit=True,
    dtype=None,
)
print(f"Model: {model.config._name_or_path}")

In [ ]:
# Cell 4: Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
model.print_trainable_parameters()

In [ ]:
# Cell 5: Load dataset and pre-format as text
# Must apply chat template BEFORE passing to SFTTrainer
# (avoids Jinja template issues with dict vs object access)
from datasets import load_dataset

raw_dataset = load_dataset("json", data_files="train.jsonl", split="train")
print(f"Raw: {len(raw_dataset)} examples")

def format_to_text(example):
    return {"text": tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False)}

dataset = raw_dataset.map(format_to_text)
print(f"Formatted: {len(dataset)} examples")
print(f"Sample (first 200 chars): {dataset[0]['text'][:200]}...")

In [ ]:
# Cell 6: Train
# - fp16 (not bf16 — T4 doesn't support bf16)
# - batch_size=1 + grad_accum=8 (fits T4 16GB VRAM)
# - max_seq_length=1024 (memory-safe for T4)
# - dataset_text_field="text" (pre-formatted in Cell 5)
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    args=SFTConfig(
        output_dir="hadith-scholar-output",
        num_train_epochs=2,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        warmup_steps=10,
        logging_steps=25,
        save_steps=250,
        fp16=True,
        optim="adamw_8bit",
        seed=42,
        max_seq_length=1024,
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"Training complete! Loss: {stats.training_loss:.4f}")

In [ ]:
# Cell 7: Quick test (with RAG context — matches runtime pattern)
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": """You are a knowledgeable Islamic scholar assistant specializing in hadith sciences.
Answer questions using ONLY the hadiths provided below as context.
Always cite the hadith number when referencing a hadith.
When relevant, mention the chain of narration (isnad) to support authenticity.
Be concise and accurate.

## Relevant Hadiths:

Hadith #1 — عُمَرَ بْنَ الْخَطَّابِ
Chain: الْحُمَيْدِيُّ (gen 10, trustworthy) → سُفْيَانُ بْنُ عُيَيْنَةَ (gen 8, trustworthy) → يَحْيَى بْنُ سَعِيدٍ (gen 6, trustworthy) → عُمَرَ بْنَ الْخَطَّابِ (gen 1, Sahabi)
Narrated Umar bin Al-Khattab: I heard Allah's Messenger saying, \"The reward of deeds depends upon the intentions.\"
"""},
    {"role": "user", "content": "Assess the chain of narration for this hadith."},
]

text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=300, temperature=0.7)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
# Cell 8: Save as HuggingFace format
# (Unsloth's built-in GGUF converter has issues with Phi-4-mini,
#  so we save as HF first, then convert manually with llama.cpp)
print("Saving merged model in HuggingFace format...")
model.save_pretrained_merged(
    "hadith-scholar-hf",
    tokenizer,
    save_method="merged_16bit",
)
print("Saved to hadith-scholar-hf/")

import os
for f in os.listdir("hadith-scholar-hf"):
    size = os.path.getsize(f"hadith-scholar-hf/{f}") / (1024**3)
    if size > 0.01:
        print(f"  {f}: {size:.2f} GB")

In [ ]:
# Cell 9: Convert to GGUF with llama.cpp
!pip install -q gguf

import os
if not os.path.exists("llama.cpp"):
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp.git
    !cd llama.cpp && pip install -q -r requirements.txt

# Convert HF → GGUF f16
!python3 llama.cpp/convert_hf_to_gguf.py hadith-scholar-hf \
    --outfile hadith-scholar-f16.gguf \
    --outtype f16

# Build quantize tool and quantize to Q4_K_M
!cd llama.cpp && cmake -B build -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -1
!cd llama.cpp && cmake --build build --target llama-quantize -j 2>&1 | tail -3
!llama.cpp/build/bin/llama-quantize hadith-scholar-f16.gguf hadith-scholar-q4km.gguf Q4_K_M

if os.path.exists("hadith-scholar-q4km.gguf"):
    size = os.path.getsize("hadith-scholar-q4km.gguf") / (1024**3)
    print(f"\nSuccess! hadith-scholar-q4km.gguf: {size:.2f} GB")
else:
    print("ERROR: Quantization failed.")

In [ ]:
# Cell 10: Download GGUF file
from google.colab import files
import os

if os.path.exists("hadith-scholar-q4km.gguf"):
    print("Downloading hadith-scholar-q4km.gguf...")
    print("\nAfter download, run on your Mac:")
    print("  mv ~/Downloads/hadith-scholar-q4km.gguf models/")
    print("  ollama create hadith-scholar -f models/Modelfile")
    print("  OLLAMA_MODEL=hadith-scholar make server")
    files.download("hadith-scholar-q4km.gguf")
else:
    print("No GGUF file found. Check Cell 9 for errors.")